<a href="https://colab.research.google.com/github/SuyashPatil-max/CDC-x-Yhills/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

!pip install xgboost
!pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 11.3 MB/s eta 0:00:00


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
print('cuda' if tf.config.list_physical_devices('GPU') else 'cpu')


cuda


# Tabular Data Prediction



In [ ]:
train = pd.read_csv('train_new.csv')
train.shape

(16209, 65)

In [ ]:
X = train.iloc[:,:-1]
Y = train.iloc[:,-1]

print(X.shape ,Y.shape)

(16209, 64) (16209,)


In [ ]:
from sklearn.preprocessing import PowerTransformer
trf = PowerTransformer(standardize = True)
Y = trf.fit_transform(Y.values.reshape(-1,1))

In [ ]:
from xgboost import XGBRegressor
xgb = XGBRegressor()

In [ ]:
xgb.fit(X ,Y)
cross_val_score(xgb , X,Y ,cv =10 ,scoring='r2').mean()

np.float64(0.8790048613039566)

In [ ]:
import optuna

In [ ]:
def objective(trial) :
  estimator = trial.suggest_int('n',5,100)
  depth = trial.suggest_int('d' ,5,20)
  lr = trial.suggest_float('lr',0.01,0.5)
  Lambda = trial.suggest_int('l',1,10)
  alpha = trial.suggest_int('al',0,10)
  child = trial.suggest_int('ch',1,5,2)
  subsample = trial.suggest_float('sub',0.7,1)
  col = trial.suggest_float('col',0.7 ,1 )

  model = XGBRegressor(n_estimators=estimator,max_depth=depth,learning_rate=lr,reg_lambda=Lambda,reg_alpha=alpha,
    min_child_weight=child,subsample=subsample,colsample_bytree=col,objective="reg:squarederror",
    random_state=42,n_jobs=-1)

  score = cross_val_score(model , X,Y ,cv =10 ,scoring='r2').mean()
  return score

In [ ]:
study = optuna.create_study(direction ='minimize')
study.optimize(objective , n_trials=20)

[I 2026-01-06 19:03:01,420] A new study created in memory with name: no-name-0c38d0a0-6eab-4e78-8b8d-abae173248c9
/tmp/ipython-input-473828992.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  child = trial.suggest_int('ch',1,5,2)
[I 2026-01-06 19:03:08,182] Trial 0 finished with value: 0.8586596889615838 and parameters: {'n': 42, 'd': 8, 'lr': 0.499181201591916, 'l': 1, 'al': 4, 'ch': 5, 'sub': 0.866560096371564, 'col': 0.7985591934028026}. Best is trial 0 with value: 0.8586596889615838.
/tmp/ipython-input-473828992.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they we

In [ ]:
op = study.trials_dataframe()
op.head()

,number,value,datetime_start,datetime_complete,duration,params_al,params_ch,params_col,params_d,params_l,params_lr,params_n,params_sub,state
0,0,0.858660,2026-01-06 19:03:01.422541,2026-01-06 19:03:08.182356,0 days 00:00:06.759815,4,5,0.798559,8,1,0.499181,42,0.866560,COMPLETE
1,1,0.882338,2026-01-06 19:03:08.184785,2026-01-06 19:03:20.931449,0 days 00:00:12.746664,9,5,0.888050,11,9,0.204545,95,0.711212,COMPLETE
2,2,0.866284,2026-01-06 19:03:20.933146,2026-01-06 19:03:31.724485,0 days 00:00:10.791339,10,1,0.870903,19,8,0.131398,43,0.780632,COMPLETE
3,3,0.859238,2026-01-06 19:03:31.726616,2026-01-06 19:03:48.470872,0 days 00:00:16.744256,6,5,0.841373,16,1,0.395811,91,0.833084,COMPLETE
4,4,0.842996,2026-01-06 19:03:48.472806,2026-01-06 19:03:58.008605,0 days 00:00:09.535799,10,1,0.797389,12,6,0.046868,62,0.761158,COMPLETE


In [ ]:
j = 0
for i in op['number'] :
  if op['value'].max() == op['value'][i] :
    j =i
    print(op.iloc[i,:])
  else :
      pass

maxcol = op.iloc[j,:]

number                                        5
value                                   0.88495
datetime_start       2026-01-06 19:03:58.012331
datetime_complete    2026-01-06 19:04:06.415704
duration                 0 days 00:00:08.403373
params_al                                     5
params_ch                                     5
params_col                             0.764883
params_d                                      7
params_l                                      3
params_lr                              0.264414
params_n                                    100
params_sub                             0.911723
state                                  COMPLETE
Name: 5, dtype: object


In [ ]:
maxcol

,5
number,5
value,0.88495
datetime_start,2026-01-06 19:03:58.012331
datetime_complete,2026-01-06 19:04:06.415704
duration,0 days 00:00:08.403373
params_al,5
params_ch,5
params_col,0.764883
params_d,7
params_l,3


In [ ]:
model = XGBRegressor(n_estimators=maxcol[11],max_depth=maxcol[8],learning_rate=maxcol[10],reg_lambda=maxcol[9],reg_alpha=maxcol[5],
    min_child_weight=maxcol[6],subsample= maxcol[12],colsample_bytree=maxcol[7],objective="reg:squarederror",
    random_state=42,n_jobs=-1,device ='cuda')
model.fit(X ,Y)
cross_val_score(model , X,Y ,cv =10 ,scoring='r2').mean()

/tmp/ipython-input-1204714650.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  model = XGBRegressor(n_estimators=maxcol[11],max_depth=maxcol[8],learning_rate=maxcol[10],reg_lambda=maxcol[9],reg_alpha=maxcol[5],
/tmp/ipython-input-1204714650.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  min_child_weight=maxcol[6],subsample= maxcol[12],colsample_bytree=maxcol[7],objective="reg:squarederror",
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [19:07:40] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to hi

np.float64(0.8847726277642411)

# Tabular + Image

In [ ]:
!unzip "/content/drive/MyDrive/Colab Notebooks/CDC_Files_to_submit/images_df.zip"
!unzip "/content/drive/MyDrive/Colab Notebooks/CDC_Files_to_submit/test_image_final.zip"

Archive:  /content/drive/MyDrive/Colab Notebooks/CDC_Files_to_submit/images_df.zip
  inflating: images_df.csv           
Archive:  /content/drive/MyDrive/Colab Notebooks/CDC_Files_to_submit/test_image_final.zip
  inflating: test_image.csv          


In [ ]:
train_image = pd.read_csv('images_df.csv')
test_image = pd.read_csv('test_image.csv')

In [ ]:
train_image.shape , test_image.shape

((16110, 2353), (5396, 2353))

In [ ]:
drop_idx = np.random.choice(np.arange(X.shape[0]), size=X.shape[0] - train_image.shape[0], replace=False)
X_new = np.delete(X, drop_idx, axis=0)
Y_new = np.delete(Y, drop_idx, axis=0)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
from sklearn.model_selection import train_test_split
image_data_processed = train_image.drop(columns=['Unnamed: 0']).to_numpy()
X_img_train, X_img_test, X_tab_train, X_tab_test, y_train, y_test = train_test_split(image_data_processed, X_new, Y_new,test_size=0.2, random_state=42)
X_train_val_img =  X_img_train.reshape(-1,28,28,3)
train_image = image_data_processed.reshape(-1, 28, 28, 3)
X_img_test = X_img_test.reshape(-1, 28, 28, 3)

In [ ]:
image_input = layers.Input(shape=(28, 28, 3), name="image_input")
x = layers.Resizing(224, 224)(image_input)

base_model = ResNet50(weights="imagenet",include_top=False,input_tensor=x)
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

image_features = layers.Dense(128, activation="relu")(x)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
tabular_input = layers.Input(shape=(64,), name="tabular_input")

y = layers.BatchNormalization()(tabular_input)
y = layers.Dense(128, activation="relu")(y)
y = layers.Dropout(0.3)(y)
y = layers.Dense(64, activation="relu")(y)

tabular_features = layers.Dense(32, activation="relu")(y)


In [ ]:
combined = layers.Concatenate()([image_features, tabular_features])

z = layers.Dense(128, activation="relu")(combined)
z = layers.Dropout(0.3)(z)
z = layers.Dense(64, activation="relu")(z)

output = layers.Dense(1, activation="linear", name="price")(z)


In [ ]:
model = models.Model(
    inputs=[image_input, tabular_input],
    outputs=output
)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 28, 28, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 224, 224,  │          0 │ image_input[0][0] │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ resizing[0][0]    │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c

 Total params: 24,201,185 (92.32 MB)

 Trainable params: 609,249 (2.32 MB)

 Non-trainable params: 23,591,936 (90.00 MB)

In [ ]:
early_stop = EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)
history = model.fit([train_image, X_new],Y_new,validation_split=0.2,epochs=10,batch_size=64,callbacks=[early_stop])

Epoch 1/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 63s 263ms/step - loss: 0.6477 - mae: 0.6200 - val_loss: 34.7345 - val_mae: 5.8433
Epoch 2/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 28s 176ms/step - loss: 0.3276 - mae: 0.4346 - val_loss: 7.8432 - val_mae: 2.7203
Epoch 3/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 30s 185ms/step - loss: 0.2706 - mae: 0.3964 - val_loss: 0.9153 - val_mae: 0.7739
Epoch 4/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 31s 188ms/step - loss: 0.2502 - mae: 0.3808 - val_loss: 0.2955 - val_mae: 0.4091
Epoch 5/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 29s 181ms/step - loss: 0.2387 - mae: 0.3686 - val_loss: 0.2013 - val_mae: 0.3402
Epoch 6/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 30s 184ms/step - loss: 0.2187 - mae: 0.3548 - val_loss: 0.2145 - val_mae: 0.3525
Epoch 7/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 30s 185ms/step - loss: 0.2128 - mae: 0.3451 - val_loss: 0.1861 - val_mae: 0.3224
Epoch 8/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 30s 184ms/step - loss: 0.2019 - mae: 0.3368 - val_loss: 0.1729 - val_mae: 0.3107
Epoch 9/10
162/162 ━━━━━━━━━━━━

In [ ]:
from sklearn.metrics import r2_score
model.evaluate([X_img_test, X_tab_test], y_test)
y_pred = model.predict([X_img_test, X_tab_test])
r2_score_test = r2_score(y_test, y_pred)
print(r2_score_test)

101/101 ━━━━━━━━━━━━━━━━━━━━ 17s 119ms/step - loss: 0.1687 - mae: 0.2967
101/101 ━━━━━━━━━━━━━━━━━━━━ 17s 127ms/step
0.839056560101296


In [ ]:
from sklearn.metrics import r2_score
model.evaluate([X_img_train, X_tab_train], y_train)
y_pred_train = model.predict([X_img_train, X_tab_train])
train_r2_score = r2_score(y_train, y_pred_train)
print(train_r2_score)

403/403 ━━━━━━━━━━━━━━━━━━━━ 37s 92ms/step - loss: 0.1448 - mae: 0.2796
403/403 ━━━━━━━━━━━━━━━━━━━━ 33s 82ms/step
0.8502278596859159


In [ ]:
test_data = pd.read_csv('test_new.csv')
test_data.shape , test_image.shape

((5404, 63), (5396, 2353))

In [ ]:
test_data.head()

,Unnamed: 0,id,sqft_living,sqft_lot,sqft_above,sqft_living15,sqft_lot15,year,month_sin,month_cos,...,grade_other,yr_renovated_0,yr_renovated_other,sqft_basement_0,sqft_basement_400,sqft_basement_500,sqft_basement_600,sqft_basement_700,sqft_basement_800,sqft_basement_other
0,0,2591820310,0.166806,0.199326,0.591864,0.755685,-0.000315,-4.463097e-14,-1.224746,0.974179,...,0,1,0,1,0,0,0,0,0,0
1,1,7974200820,1.009472,-0.254074,0.308057,0.730493,-0.353140,-4.463097e-14,-1.224746,-0.435512,...,0,1,0,0,0,0,0,0,0,1
2,2,7701450110,1.697622,0.559292,1.899235,2.009726,0.435429,-4.463097e-14,-1.224746,-0.435512,...,0,1,0,1,0,0,0,0,0,0
3,3,9522300010,2.034593,1.129022,1.945594,2.009726,1.266848,9.242607e-14,1.413556,0.337955,...,1,1,0,1,0,0,0,0,0,0
4,4,9510861140,0.682605,-0.587696,1.060206,0.573921,-1.019664,-4.463097e-14,-0.697348,-1.083843,...,0,1,0,1,0,0,0,0,0,0


In [ ]:
drop_idx = np.random.choice(np.arange(test_data.shape[0]), size=test_data.shape[0] - test_image.shape[0], replace=False)
test_new = np.delete(test_data, drop_idx, axis=0)

In [ ]:
test_new.shape

(5396, 63)

In [ ]:
test_cols = test_data.columns
test_cols

Index(['Unnamed: 0', 'id', 'sqft_living', 'sqft_lot', 'sqft_above',
       'sqft_living15', 'sqft_lot15', 'year', 'month_sin', 'month_cos',
       'dow_sin', 'dow_cos', 'floors', 'yr_built', 'zipcode', 'lat', 'long',
       'bedrooms_1', 'bedrooms_2', 'bedrooms_3', 'bedrooms_4', 'bedrooms_5',
       'bedrooms_6', 'bedrooms_other', 'bathrooms_1.0', 'bathrooms_1.5',
       'bathrooms_1.75', 'bathrooms_2.0', 'bathrooms_2.25', 'bathrooms_2.5',
       'bathrooms_2.75', 'bathrooms_3.0', 'bathrooms_3.25', 'bathrooms_3.5',
       'bathrooms_other', 'waterfront_0', 'waterfront_1', 'view_0', 'view_1',
       'view_2', 'view_3', 'view_4', 'condition_1', 'condition_2',
       'condition_3', 'condition_4', 'condition_5', 'grade_10', 'grade_11',
       'grade_6', 'grade_7', 'grade_8', 'grade_9', 'grade_other',
       'yr_renovated_0', 'yr_renovated_other', 'sqft_basement_0',
       'sqft_basement_400', 'sqft_basement_500', 'sqft_basement_600',
       'sqft_basement_700', 'sqft_basement_800', 'sqft_b

In [ ]:
test_new = pd.DataFrame(test_new ,columns = test_cols)

In [ ]:
test_new.head()

,Unnamed: 0,id,sqft_living,sqft_lot,sqft_above,sqft_living15,sqft_lot15,year,month_sin,month_cos,...,grade_other,yr_renovated_0,yr_renovated_other,sqft_basement_0,sqft_basement_400,sqft_basement_500,sqft_basement_600,sqft_basement_700,sqft_basement_800,sqft_basement_other
0,0.0,2.591820e+09,0.166806,0.199326,0.591864,0.755685,-0.000315,-4.463097e-14,-1.224746,0.974179,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,7.974201e+09,1.009472,-0.254074,0.308057,0.730493,-0.353140,-4.463097e-14,-1.224746,-0.435512,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2.0,7.701450e+09,1.697622,0.559292,1.899235,2.009726,0.435429,-4.463097e-14,-1.224746,-0.435512,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3.0,9.522300e+09,2.034593,1.129022,1.945594,2.009726,1.266848,9.242607e-14,1.413556,0.337955,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4.0,9.510861e+09,0.682605,-0.587696,1.060206,0.573921,-1.019664,-4.463097e-14,-0.697348,-1.083843,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
test_new = test_new.drop('Unnamed: 0',axis =1 )

In [ ]:
test_id = test_new['id']
test_new = test_new.drop(['id'],axis = 1)

In [ ]:
test_new.head()

,sqft_living,sqft_lot,sqft_above,sqft_living15,sqft_lot15,year,month_sin,month_cos,dow_sin,dow_cos,...,grade_other,yr_renovated_0,yr_renovated_other,sqft_basement_0,sqft_basement_400,sqft_basement_500,sqft_basement_600,sqft_basement_700,sqft_basement_800,sqft_basement_other
0,0.166806,0.199326,0.591864,0.755685,-0.000315,-4.463097e-14,-1.224746,0.974179,-0.769838,1.329658,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.009472,-0.254074,0.308057,0.730493,-0.353140,-4.463097e-14,-1.224746,-0.435512,0.021347,-1.153011,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1.697622,0.559292,1.899235,2.009726,0.435429,-4.463097e-14,-1.224746,-0.435512,-1.387319,-1.153011,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2.034593,1.129022,1.945594,2.009726,1.266848,9.242607e-14,1.413556,0.337955,0.782134,0.889491,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.682605,-0.587696,1.060206,0.573921,-1.019664,-4.463097e-14,-0.697348,-1.083843,-0.769838,1.329658,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
X_img_for_pred = test_image.drop(columns=['Unnamed: 0']).to_numpy().reshape(-1, 28, 28, 3)

X_tab_cols = X.columns.tolist()

test_data_filtered_rows = test_data.drop(test_data.index[drop_idx])

X_tab_for_pred_df = test_data_filtered_rows.reindex(columns=X_tab_cols, fill_value=0)
X_tab_for_pred = X_tab_for_pred_df.to_numpy()

y_pred = model.predict([X_img_for_pred, X_tab_for_pred])
y_pred = y_pred.reshape(-1)

169/169 ━━━━━━━━━━━━━━━━━━━━ 13s 78ms/step


In [ ]:
y_pred.shape , test_id.shape

((5396,), (5396,))

In [ ]:
test_id

,id
0,2.591820e+09
1,7.974201e+09
2,7.701450e+09
3,9.522300e+09
4,9.510861e+09
...,...
5391,7.732500e+09
5392,3.856904e+09
5393,2.557000e+09
5394,4.386700e+09


In [ ]:
y_pred_real = trf.inverse_transform(y_pred.reshape(-1, 1)).ravel()
y_pred_real.shape

(5396,)

In [ ]:
submission = pd.DataFrame({"id":test_id,"price": y_pred_real})
submission.to_csv("submission.csv", index=False)

print("submission.csv saved")
print(submission.head())

submission.csv saved
             id         price
0  2.591820e+09  3.517812e+05
1  7.974201e+09  7.066524e+05
2  7.701450e+09  1.170471e+06
3  9.522300e+09  1.712728e+06
4  9.510861e+09  6.020632e+05
